# Download OpenET Data via Google Earth Engine

Downloads bimonthly median ET for Iowa over 2019–2023.

**Workflow:**
1. Load OpenET daily ensemble from GEE
2. Mask water bodies, low-quality pixels, and implausible values
3. Aggregate to bimonthly medians (1–15th and 16th–end of month)
4. Reproject/resample to match a user-uploaded reference raster
5. Download each time step directly to disk (no Google Drive needed)

**Prerequisites:**
- A Google Earth Engine account: https://earthengine.google.com/
- `earthengine-api` and `requests` installed: `pip install earthengine-api requests`
- Run `earthengine authenticate` once in a terminal to set up credentials
- Your reference raster uploaded to GEE Assets (see Section 4 below)


---
## Section 0 — Authenticate & Initialize

In [1]:
import ee
import datetime
import calendar
import time
from collections import Counter

# ── First time only: uncomment and run this line to authenticate ──
# ee.Authenticate()

# Replace 'your-project-id' with your GEE Cloud project ID.
# Find it at: https://console.cloud.google.com/
ee.Initialize(project='your-project-id')
print('Earth Engine initialized successfully.')

ModuleNotFoundError: No module named 'ee'

---
## Section 1 — Configuration

**Edit the values in this cell.** Everything else should run without changes.

In [ ]:
# ── Years to process ─────────────────────────────────────────────────────────
YEARS = list(range(2019, 2024))   # 2019 through 2023 inclusive

# ── OpenET GEE collection IDs ────────────────────────────────────────────────
# Daily ensemble — mean of all 6 OpenET models (SSEBop, SIMS, geeSEBAL, etc.)
OPENET_DAILY = 'OpenET/ENSEMBLE/CONUS/GRIDMET/DAILY/v2_0'

# ── Target grid: SIF OCO-2 0.05° ─────────────────────────────────────────────
# All outputs are aligned to the global SIF OCO-2 grid (Yu et al. 2019).
# Grid: WGS84, 0.05° resolution, cell centers at multiples of 0.05° offset by 0.025°
# (e.g. lon: -179.975, -179.925, ... lat: 89.975, 89.925, ...)
#
# The crsTransform encodes the upper-left corner of the global grid:
#   [xScale, xShear, xOrigin, yShear, yScale, yOrigin]
TARGET_CRS       = 'EPSG:4326'
TARGET_TRANSFORM = [0.05, 0, -180.0, 0, -0.05, 90.0]  # global SIF grid origin

# ── Local output directory ────────────────────────────────────────────────────
from pathlib import Path
OUTPUT_DIR = Path().resolve().parent.parent.parent / 'data' / 'raw' / 'OpenET'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

# ── Band names in the OpenET daily ensemble ──────────────────────────────────
ET_BAND    = 'et_ensemble_mad'       # median absolute deviation ensemble mean (mm/day)
COUNT_BAND = 'et_ensemble_mad_count'  # number of models contributing (quality proxy)

# ── Quality filter: minimum number of models that must agree ─────────────────
MIN_MODEL_COUNT = 3


---
## Section 2 — Inspect the OpenET Collection

Run this first to confirm band names, date range, and image count before processing.

In [ ]:
# Load and inspect the daily ensemble collection
openet_col = ee.ImageCollection(OPENET_DAILY)

# Grab one image for inspection
sample = openet_col.first()

print('=== OpenET Daily Ensemble ===')
print('Band names  :', sample.bandNames().getInfo())
print('Properties  :', sample.propertyNames().getInfo())
print('First image :', ee.Date(sample.get('system:time_start')).format('YYYY-MM-dd').getInfo())

# Check how many images are available over Iowa for 2015–2020
iowa = ee.FeatureCollection('TIGER/2018/States') \
         .filter(ee.Filter.eq('NAME', 'Iowa')).geometry()

n_images = openet_col.filterBounds(iowa) \
                     .filterDate('2015-01-01', '2021-01-01') \
                     .size().getInfo()
print(f'Images over Iowa (2015–2020): {n_images}')

# Print scale/resolution of the native data
print('Native scale (m):', sample.select(0).projection().nominalScale().getInfo())

---
## Section 3 — Study Area (Iowa)

In [ ]:
# Load Iowa state boundary from the US Census TIGER dataset
states = ee.FeatureCollection('TIGER/2018/States')
iowa   = states.filter(ee.Filter.eq('NAME', 'Iowa')).geometry()

# Quick sanity check
print('Iowa bounds:', iowa.bounds().getInfo()['coordinates'])

---
## Section 5 — Masking Functions

In [ ]:
def mask_low_quality(image, count_band=COUNT_BAND):
    """
    Retain only pixels where at least MIN_MODEL_COUNT of the 6 ensemble
    members produced a valid ET estimate.

    Note: if the daily collection does not have a 'count' band, comment
    this mask out and rely on the ET range filter instead.
    """
    band_names = image.bandNames()
    has_count  = band_names.contains(count_band)

    def apply_count_mask(img):
        return img.updateMask(img.select(count_band).gte(MIN_MODEL_COUNT))

    # Only apply if the band exists (avoids errors if the collection differs)
    return ee.Image(ee.Algorithms.If(has_count, apply_count_mask(image), image))

def apply_masks(image):
    """Convenience wrapper — apply all three masks in sequence."""
    image = mask_low_quality(image)
    return image

print('Masking function defined.')

---
## Section 6 — Bimonthly Period Generator

In [ ]:
def get_bimonthly_periods(years):
    """
    Return a list of dicts, one per bimonthly period.
    Each month is split into:
      - First half : days  1–15
      - Second half: days 16–end-of-month

    Each dict has:
      'start'      : inclusive start date string (YYYY-MM-DD)
      'end'        : inclusive end date string
      'end_excl'   : exclusive end date (for GEE filterDate, which is [start, end) )
      'label'      : compact string used in filenames, e.g. '20150101_20150115'
    """
    periods = []
    for year in years:
        for month in range(1, 13):
            last_day = calendar.monthrange(year, month)[1]

            # --- First half: 1st to 15th ---
            s1 = datetime.date(year, month, 1)
            e1 = datetime.date(year, month, 15)
            periods.append({
                'start'    : s1.strftime('%Y-%m-%d'),
                'end'      : e1.strftime('%Y-%m-%d'),
                'end_excl' : (e1 + datetime.timedelta(days=1)).strftime('%Y-%m-%d'),
                'label'    : f'{s1.strftime("%Y%m%d")}_{e1.strftime("%Y%m%d")}'
            })

            # --- Second half: 16th to end of month ---
            s2 = datetime.date(year, month, 16)
            e2 = datetime.date(year, month, last_day)
            periods.append({
                'start'    : s2.strftime('%Y-%m-%d'),
                'end'      : e2.strftime('%Y-%m-%d'),
                'end_excl' : (e2 + datetime.timedelta(days=1)).strftime('%Y-%m-%d'),
                'label'    : f'{s2.strftime("%Y%m%d")}_{e2.strftime("%Y%m%d")}'
            })

    return periods


periods = get_bimonthly_periods(YEARS)
print(f'Total bimonthly periods: {len(periods)}')
print('\nFirst 6 periods:')
for p in periods[:6]:
    print(f"  {p['start']}  →  {p['end']}   label: {p['label']}")

---
## Section 7 — Processing Function

For each bimonthly period:
1. Filter the daily OpenET collection to that date window
2. Apply masks per-image before aggregating (avoids bad values pulling the median)
3. Compute pixel-wise **median** ET across all valid daily images
4. Also compute the **mean model count** as a spatial quality indicator
5. Clip to Iowa

Reprojection happens at export time (more efficient than `.reproject()` on the image).

In [ ]:
def process_period(period, collection, apply_masks=True):
    """
    Compute bimonthly median ET for one period.

    Parameters
    ----------
    period       : dict from get_bimonthly_periods()
    collection   : ee.ImageCollection — the full OpenET daily collection
    apply_masks  : bool — whether to apply water / quality / range masks

    Returns
    -------
    ee.Image with two bands:
      'et'               — median daily ET (mm/day) over the period
      'mean_model_count' — average number of models contributing (quality proxy)
    """
    # GEE filterDate uses [start, end) — 'end_excl' ensures the last day is included
    filtered = (
        collection
        .filterDate(period['start'], period['end_excl'])
        .filterBounds(iowa)
    )
    
    # Apply masks to every image before aggregating
    if apply_masks:
        filtered = filtered.map(apply_masks)

    # Pixel-wise median ET across all valid daily images in the window
    median_et = filtered.select(ET_BAND).median().rename(ET_BAND)

    # Mean model count — higher value = more models agreed = higher quality
    # (If COUNT_BAND doesn't exist in your collection, comment out these two lines)
    mean_count = filtered.select(COUNT_BAND).mean().rename('mean_model_count')

    # Combine into a single multi-band image and clip to Iowa
    result = median_et.addBands(mean_count).clip(iowa)

    # Store dates as image properties (handy metadata)
    result = result.set({
        'period_start': period['start'],
        'period_end'  : period['end'],
        'label'       : period['label']
    })

    return result


print('process_period() defined.')

---
## Section 8 — Test 

In [ ]:
openet_collection = ee.ImageCollection(OPENET_DAILY)

test_period = periods[0]   # first bimonthly period: Jan 1–15, 2015
print(f"Testing: {test_period['start']} → {test_period['end']}")

test_image = process_period(test_period, openet_collection)

print('Output bands:', test_image.bandNames().getInfo())

# Compute summary stats over Iowa (uses a coarse scale for speed)
# This call hits GEE servers — may take ~30–60 seconds
stats = test_image.select(ET_BAND).reduceRegion(
    reducer  = ee.Reducer.mean()
                         .combine(ee.Reducer.minMax(), sharedInputs=True)
                         .combine(ee.Reducer.count(),  sharedInputs=True),
    geometry = iowa,
    scale    = 500,        # coarse resolution for a quick check
    maxPixels= 1e9
)
print('ET stats for test period:')
for k, v in stats.getInfo().items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

---
## Section 9 — Download Loop

Downloads each bimonthly period directly to disk using GEE's `getDownloadURL()`.
No Google Drive required — files are written straight to `data/raw/OpenET/`.

Already-downloaded periods are skipped automatically (safe to re-run).

In [ ]:
import requests
import zipfile
import io

openet_collection = ee.ImageCollection(OPENET_DAILY)

# Skip already-downloaded files (resume support)
existing = {f.stem for f in OUTPUT_DIR.glob('OpenET_Iowa_*.tif')}
print(f'{len(existing)} files already downloaded — will skip these.')

failed = []

for i, period in enumerate(periods):
    filename = f'OpenET_Iowa_{period["label"]}'
    out_path = OUTPUT_DIR / f'{filename}.tif'

    if filename in existing:
        print(f'[{i+1:3d}/{len(periods)}] Skipped (exists): {filename}')
        continue

    try:
        image = process_period(period, openet_collection)

        # Download directly from GEE to disk — aligned to SIF OCO-2 0.05° grid.
        # TARGET_TRANSFORM pins the output to the global SIF grid origin so pixels
        # snap exactly to SIF, ECOSTRESS, and NLDAS outputs.
        url = image.select(ET_BAND).getDownloadURL({
            'region'      : iowa,
            'crs'         : TARGET_CRS,
            'crsTransform': TARGET_TRANSFORM,
            'fileFormat'  : 'GeoTIFF',
        })

        response = requests.get(url, timeout=300)
        response.raise_for_status()

        # GEE wraps the GeoTIFF in a zip file — extract it
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            tif_names = [n for n in zf.namelist() if n.endswith('.tif')]
            with open(out_path, 'wb') as f:
                f.write(zf.read(tif_names[0]))

        print(f'[{i+1:3d}/{len(periods)}] Saved: {out_path.name}')

    except Exception as e:
        print(f'[{i+1:3d}/{len(periods)}] FAILED: {filename} — {e}')
        failed.append(period['label'])
        time.sleep(2)

print(f'\nDone. {len(periods) - len(failed)} downloaded, {len(failed)} failed.')
if failed:
    print('Failed periods (re-run to retry):', failed)
